# P5 — Elexon Generation Data: Exploration Phase

Purpose: establish what is actually available from the Elexon Insights Solution API and the OSUKED Power-Station-Dictionary before building the full extraction pipeline. This notebook verifies column names, join keys, and per-site data coverage against the live sources — nothing here is assumed from documentation alone.

Scope note: this is exploration only. No site-inclusion filters, minimum history length, or final storage design are decided in this notebook — those are for Simon to review and decide based on the findings below.

In [1]:
import requests
import pandas as pd
from pathlib import Path
from io import StringIO
import time

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

REF_DIR = Path("../data/reference")
REF_DIR.mkdir(parents=True, exist_ok=True)

OSUKED_RAW = "https://raw.githubusercontent.com/OSUKED/Power-Station-Dictionary/main"
ELEXON_API = "https://data.elexon.co.uk/bmrs/api/v1"

## Step 2 — OSUKED Power-Station-Dictionary: pull and inspect

The brief names the two files as `bmu-fuel-types.csv` and `plant-locations.csv`. The live repo tree (checked via the GitHub API) does not use those exact filenames or that flat layout — the current paths are:

- `data/linked-datapackages/bmu-fuel-types/fuel_types.csv`
- `data/linked-datapackages/plant-locations/plant-locations.csv`

There is also a third file, `data/dictionary/ids.csv`, which turns out to be essential — see the join-key investigation below.

In [2]:
osuked_files = {
    "fuel_types": "data/linked-datapackages/bmu-fuel-types/fuel_types.csv",
    "plant_locations": "data/linked-datapackages/plant-locations/plant-locations.csv",
    "dictionary_ids": "data/dictionary/ids.csv",
}

osuked = {}
for name, path in osuked_files.items():
    resp = requests.get(f"{OSUKED_RAW}/{path}", timeout=30)
    resp.raise_for_status()
    df = pd.read_csv(StringIO(resp.text))
    df.to_csv(REF_DIR / f"osuked_{name}.csv", index=False)
    osuked[name] = df
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols -> saved to data/reference/osuked_{name}.csv")
    print(list(df.columns))
    print()

fuel_types: 462 rows, 3 cols -> saved to data/reference/osuked_fuel_types.csv
['ngc_bmu_id', 'fuel_type', 'comments']

plant_locations: 243 rows, 3 cols -> saved to data/reference/osuked_plant_locations.csv
['dictionary_id', 'longitude', 'latitude']



dictionary_ids: 277 rows, 19 cols -> saved to data/reference/osuked_dictionary_ids.csv
['dictionary_id', 'gppd_idnr', 'esail_id', 'name', 'sett_bmu_id', 'ngc_bmu_id', '4c_offshore_id', 'windpowernet_id', 'wikidata_id', 'wikipedia_id', 'power_technology_id', 'eutl_id', 'eic_id', 'cfd_id', 'jrc_id', 'iaea_id', 'old_repd_id', 'new_repd_id', 'crown_estate_id']



In [3]:
for name, df in osuked.items():
    print(f"--- {name} ---")
    display(df.head(3))

--- fuel_types ---


,ngc_bmu_id,fuel_type,comments
0,ABRBO-1,WIND,NaN
1,ABRTW-1,WIND,NaN
2,ACHLW-1,WIND,NaN


--- plant_locations ---


,dictionary_id,longitude,latitude
0,10000,-3.603516,57.480403
1,10001,-1.267570,51.623630
2,10002,-3.404866,51.387312


--- dictionary_ids ---


,dictionary_id,gppd_idnr,esail_id,name,sett_bmu_id,ngc_bmu_id,4c_offshore_id,windpowernet_id,wikidata_id,wikipedia_id,power_technology_id,eutl_id,eic_id,cfd_id,jrc_id,iaea_id,old_repd_id,new_repd_id,crown_estate_id
0,10000,NaN,MARK,Rothes Bio-Plant CHP,"E_MARK-1, E_MARK-2","MARK-1, MARK-2",NaN,NaN,NaN,NaN,NaN,NaN,48W000000MARK-1D,NaN,NaN,NaN,NaN,NaN,NaN
1,10001,"GBR1000377, GBR1000369",DIDC,Didcot,"T_DIDC1, T_DIDC2, T_DIDC4, T_DIDC3, T_DIDC1G, ...","DIDC1, DIDC2, DIDC4, DIDC3, DIDC1G, DIDC2G, DI...",NaN,NaN,Q3298465,Didcot_power_stations,NaN,97165,"48W00000DIDC01G1, 48W00000DIDC02GZ, 48W00000DI...",NaN,NaN,NaN,NaN,NaN,NaN
2,10002,"GBR1000374, GBR1000375",ABTH,Aberthaw B,"T_ABTH7, T_ABTH8, T_ABTH9, T_ABTH7G, T_ABTH8G,...","ABTH7, ABTH8, ABTH9, ABTH7G, ABTH8G, ABTH9G",NaN,NaN,Q4667192,Aberthaw_power_stations,NaN,97175,"48W0000000ABTH7Y, 48W0000000ABTH8W, 48W0000000...",NaN,NaN,NaN,NaN,NaN,NaN


### Join-key investigation: `plant_locations` → `fuel_types`

`plant_locations` is keyed on `dictionary_id` only (plus lat/long) — no BMU id. `fuel_types` is keyed on `ngc_bmu_id`. There is no direct shared column between the two.

`dictionary_ids` is the intermediate lookup: it carries `dictionary_id` alongside `sett_bmu_id` and `ngc_bmu_id`. So the join path is:

```
plant_locations.dictionary_id -> dictionary_ids.dictionary_id -> dictionary_ids.ngc_bmu_id -> fuel_types.ngc_bmu_id
```

Complication worth flagging: in `dictionary_ids`, `ngc_bmu_id` (and `sett_bmu_id`) are **comma-separated lists**, not single values — one dictionary/site entry (e.g. a multi-unit power station like Didcot) maps to several individual BMUs. That needs an explode/split step before any join, and it means the site→BMU relationship is one-to-many, not one-to-one.

In [4]:
# Explode dictionary_ids.ngc_bmu_id (comma-separated) into one row per BMU
ids = osuked["dictionary_ids"][["dictionary_id", "name", "ngc_bmu_id", "sett_bmu_id"]].copy()
ids = ids.dropna(subset=["ngc_bmu_id"])
ids["ngc_bmu_id"] = ids["ngc_bmu_id"].str.split(",")
ids_exploded = ids.explode("ngc_bmu_id")
ids_exploded["ngc_bmu_id"] = ids_exploded["ngc_bmu_id"].str.strip()

print(f"dictionary_ids: {len(osuked['dictionary_ids'])} site rows -> {len(ids_exploded)} exploded BMU rows")

# dictionary_id -> lat/long
site_locations = osuked["plant_locations"]

# Join: locations -> exploded ids -> fuel_types, all on the confirmed key path
linked = (
    site_locations.merge(ids_exploded, on="dictionary_id", how="inner")
    .merge(osuked["fuel_types"], on="ngc_bmu_id", how="inner")
)

print(f"plant_locations sites: {len(site_locations)}")
print(f"fuel_types BMUs: {len(osuked['fuel_types'])}")
print(f"Rows successfully linked location -> BMU -> fuel type: {len(linked)}")
print(f"Distinct sites (dictionary_id) represented in the link: {linked['dictionary_id'].nunique()}")
linked.head(10)

dictionary_ids: 277 site rows -> 444 exploded BMU rows
plant_locations sites: 243
fuel_types BMUs: 462
Rows successfully linked location -> BMU -> fuel type: 403
Distinct sites (dictionary_id) represented in the link: 213


,dictionary_id,longitude,latitude,name,ngc_bmu_id,sett_bmu_id,fuel_type,comments
0,10000,-3.603516,57.480403,Rothes Bio-Plant CHP,MARK-1,"E_MARK-1, E_MARK-2",BIOMASS,NaN
1,10000,-3.603516,57.480403,Rothes Bio-Plant CHP,MARK-2,"E_MARK-1, E_MARK-2",BIOMASS,NaN
2,10001,-1.267570,51.623630,Didcot,DIDC01G,"T_DIDC1, T_DIDC2, T_DIDC4, T_DIDC3, T_DIDC1G, ...",OCGT,NaN
3,10001,-1.267570,51.623630,Didcot,DIDC02G,"T_DIDC1, T_DIDC2, T_DIDC4, T_DIDC3, T_DIDC1G, ...",OCGT,NaN
4,10001,-1.267570,51.623630,Didcot,DIDC03G,"T_DIDC1, T_DIDC2, T_DIDC4, T_DIDC3, T_DIDC1G, ...",OCGT,NaN
5,10001,-1.267570,51.623630,Didcot,DIDC04G,"T_DIDC1, T_DIDC2, T_DIDC4, T_DIDC3, T_DIDC1G, ...",OCGT,NaN
6,10001,-1.267570,51.623630,Didcot,DIDCB5,"T_DIDC1, T_DIDC2, T_DIDC4, T_DIDC3, T_DIDC1G, ...",CCGT,NaN
7,10001,-1.267570,51.623630,Didcot,DIDCB6,"T_DIDC1, T_DIDC2, T_DIDC4, T_DIDC3, T_DIDC1G, ...",CCGT,NaN
8,10002,-3.404866,51.387312,Aberthaw B,ABTH7,"T_ABTH7, T_ABTH8, T_ABTH9, T_ABTH7G, T_ABTH8G,...",COAL,NaN
9,10002,-3.404866,51.387312,Aberthaw B,ABTH8,"T_ABTH7, T_ABTH8, T_ABTH9, T_ABTH7G, T_ABTH8G,...",COAL,NaN


## Step 3 — Elexon Insights Solution: live BMU reference list

Querying `/reference/bmunits/all` on the public Insights Solution API (`data.elexon.co.uk/bmrs/api/v1`) — this is the current, live list of BMUs, independent of the OSUKED dictionary.

In [5]:
resp = requests.get(f"{ELEXON_API}/reference/bmunits/all", timeout=60)
resp.raise_for_status()
bmunits = pd.DataFrame(resp.json())
bmunits.to_csv(REF_DIR / "elexon_bmunits_all.csv", index=False)

print(f"bmunits/all: {bmunits.shape[0]} rows, {bmunits.shape[1]} cols")
print(list(bmunits.columns))
bmunits.head(3)

bmunits/all: 3053 rows, 22 cols
['nationalGridBmUnit', 'elexonBmUnit', 'eic', 'fuelType', 'leadPartyName', 'bmUnitType', 'fpnFlag', 'bmUnitName', 'leadPartyId', 'demandCapacity', 'generationCapacity', 'productionOrConsumptionFlag', 'transmissionLossFactor', 'workingDayCreditAssessmentImportCapability', 'nonWorkingDayCreditAssessmentImportCapability', 'workingDayCreditAssessmentExportCapability', 'nonWorkingDayCreditAssessmentExportCapability', 'creditQualifyingStatus', 'demandInProductionFlag', 'gspGroupId', 'gspGroupName', 'interconnectorId']


,nationalGridBmUnit,elexonBmUnit,eic,fuelType,leadPartyName,bmUnitType,fpnFlag,bmUnitName,leadPartyId,demandCapacity,generationCapacity,productionOrConsumptionFlag,transmissionLossFactor,workingDayCreditAssessmentImportCapability,nonWorkingDayCreditAssessmentImportCapability,workingDayCreditAssessmentExportCapability,nonWorkingDayCreditAssessmentExportCapability,creditQualifyingStatus,demandInProductionFlag,gspGroupId,gspGroupName,interconnectorId
0,ABERU-1,E_ABERDARE,NaN,NaN,UK Power Reserve Limited,E,True,Aberdare Power Station,UKPR,0.000,15.400,C,0.0167661,0.000,0.000,6.160,6.160,True,False,_K,South Wales,NaN
1,ABRBO-1,T_ABRBO-1,48W00000ABRBO-19,WIND,Aberdeen Offshore Wind Farm,T,True,ABRBO-1,ABERDEEN,-2.000,99.000,P,-0.0200357,-0.800,-0.800,39.600,39.600,True,False,NaN,NaN,NaN
2,ABRTW-1,E_ABRTW-1,48W00000ABRTW-1Z,WIND,Npower Commercial Gas Limited,E,True,Auchrobert Wind Farm,NPOWER02,-10.000,36.000,C,-0.0066390,-4.000,-4.000,14.400,14.400,True,False,_N,South Scotland,NaN


### Cross-check: Elexon `nationalGridBmUnit` vs OSUKED `ngc_bmu_id`

The field names differ (`nationalGridBmUnit` vs `ngc_bmu_id`) but on inspection they carry the same identifier format (e.g. `ABERU-1`), so this is the shared key between the live Elexon list and the OSUKED fuel-type table.

In [6]:
elexon_bmu_ids = set(bmunits["nationalGridBmUnit"].dropna().str.strip())
osuked_bmu_ids = set(osuked["fuel_types"]["ngc_bmu_id"].dropna().str.strip())

only_elexon = elexon_bmu_ids - osuked_bmu_ids
only_osuked = osuked_bmu_ids - elexon_bmu_ids
in_both = elexon_bmu_ids & osuked_bmu_ids

print(f"Elexon live BMU list: {len(elexon_bmu_ids)} distinct nationalGridBmUnit values")
print(f"OSUKED fuel_types list: {len(osuked_bmu_ids)} distinct ngc_bmu_id values")
print(f"In both: {len(in_both)}")
print(f"Only in Elexon (no OSUKED fuel type): {len(only_elexon)}")
print(f"Only in OSUKED (not in live Elexon list): {len(only_osuked)}")
print()
print("Sample only-in-OSUKED (may be decommissioned/stale):", sorted(only_osuked)[:10])
print("Sample only-in-Elexon (no fuel-type mapping yet):", sorted(only_elexon)[:10])

Elexon live BMU list: 3052 distinct nationalGridBmUnit values
OSUKED fuel_types list: 462 distinct ngc_bmu_id values
In both: 397
Only in Elexon (no OSUKED fuel type): 2655
Only in OSUKED (not in live Elexon list): 65

Sample only-in-OSUKED (may be decommissioned/stale): ['ABTH7', 'ABTH7G', 'ABTH8', 'ABTH8G', 'ABTH9', 'ABTH9G', 'BARK-1', 'BARKB2', 'BRYP-1', 'CAIRW-1']
Sample only-in-Elexon (no fuel-type mapping yet): ['AG-ADL00B', 'AG-ADL01M', 'AG-ADL02G', 'AG-ADL03E', 'AG-ADL04H', 'AG-ADL05H', 'AG-AECO01', 'AG-AEDF01', 'AG-AEDF02', 'AG-AEDF03']


In [7]:
# Elexon's own bmunits/all response carries a fuelType field too — check how populated it is
fuel_populated = bmunits["fuelType"].notna().sum()
print(f"Elexon bmunits/all rows with a non-null fuelType: {fuel_populated} / {len(bmunits)}")
bmunits["fuelType"].value_counts(dropna=False)

Elexon bmunits/all rows with a non-null fuelType: 579 / 3053


fuelType
NaN        2474
WIND        278
OTHER       101
CCGT         65
NPSHYD       36
OCGT         26
BIOMASS      18
PS           16
NUCLEAR      16
COAL         10
INTELEC       2
INTGRNL       2
INTVKL        2
INTNED        1
INTEW         1
INTFR         1
INTIFA2       1
INTIRL        1
INTNSL        1
INTNEM        1
Name: count, dtype: int64

## Step 4 — B1610 stream: history depth for a cross-fuel-type sample

Confirmed this is the public Insights Solution **stream** endpoint (`/datasets/B1610/stream`), not the legacy key-gated BMRS API — no API key was sent or required for any call in this notebook.

Working query parameters (found by probing, since the rendered API docs page is JS-only and didn't return static content): `from`, `to`, `bmUnit`. The `settlementDateFrom`/`settlementDateTo`/`publishDateTimeFrom` variants named in some docs excerpts returned 404 — `from`/`to` is what the live endpoint actually accepts.

Approach for "roughly how far back": rather than pulling full history per BMU (potentially tens of thousands of half-hourly rows each), we binary-search for the earliest date with any returned records, using narrow 1-day probe windows. This is a coarse, day-level estimate — good enough to spot the wind-vs-thermal pattern without a full pull.

In [8]:
osuked["fuel_types"]["fuel_type"].value_counts()

fuel_type
WIND             193
CCGT              70
COAL              55
OCGT              45
NPSHYD            36
NUCLEAR           20
BIOMASS           19
PS                16
RECIPROCATING      3
BATTERY            2
Wind               1
TIDAL              1
CHP                1
Name: count, dtype: int64

In [9]:
# Pick one BMU per major fuel type, restricted to BMUs confirmed live in Elexon's own list (in_both)
candidates = osuked["fuel_types"][osuked["fuel_types"]["ngc_bmu_id"].isin(in_both)]

target_fuel_types = ["WIND", "CCGT", "NUCLEAR", "COAL", "BIOMASS", "NPSHYD"]
sample = []
for ft in target_fuel_types:
    rows = candidates[candidates["fuel_type"] == ft]
    if len(rows):
        sample.append(rows.iloc[0])

sample_df = pd.DataFrame(sample).reset_index(drop=True)
sample_df[["ngc_bmu_id", "fuel_type"]]

,ngc_bmu_id,fuel_type
0,ABRBO-1,WIND
1,BAGE-1,CCGT
2,DNGB21,NUCLEAR
3,DRAXX-5,COAL
4,DNBAR-1,BIOMASS
5,CAS-BEU01,NPSHYD


In [10]:
from datetime import date, timedelta

def has_data(bmu, probe_date):
    params = {
        "from": probe_date.isoformat(),
        "to": (probe_date + timedelta(days=1)).isoformat(),
        "bmUnit": bmu,
    }
    r = requests.get(f"{ELEXON_API}/datasets/B1610/stream", params=params, timeout=30)
    r.raise_for_status()
    return len(r.json()) > 0

def earliest_data_date(bmu, lo, hi, max_iter=14):
    """Binary search for the earliest date with B1610 data, day-level resolution.
    Assumes data presence is monotonic in time (true for commissioning-onward generation).
    """
    if not has_data(bmu, hi):
        return None  # no data even at the recent end - unexpected, flag it
    if has_data(bmu, lo):
        return f"<= {lo.isoformat()} (data present at search floor, could go back further)"
    for _ in range(max_iter):
        if (hi - lo).days <= 1:
            break
        mid = lo + (hi - lo) / 2
        if has_data(bmu, mid):
            hi = mid
        else:
            lo = mid
    return hi.isoformat()

SEARCH_LO = date(2005, 1, 1)
SEARCH_HI = date.today() - timedelta(days=10)  # allow for the ~5 working day publication lag

results = []
for _, row in sample_df.iterrows():
    bmu = row["ngc_bmu_id"]
    earliest = earliest_data_date(bmu, SEARCH_LO, SEARCH_HI)
    results.append({"ngc_bmu_id": bmu, "fuel_type": row["fuel_type"], "earliest_data_date": earliest})
    print(f"{row['fuel_type']:10s} {bmu:12s} -> earliest data ~ {earliest}")

history_df = pd.DataFrame(results)
history_df

WIND       ABRBO-1      -> earliest data ~ 2019-01-31
CCGT       BAGE-1       -> earliest data ~ None


NUCLEAR    DNGB21       -> earliest data ~ 2019-01-31


COAL       DRAXX-5      -> earliest data ~ 2019-01-31


BIOMASS    DNBAR-1      -> earliest data ~ 2021-02-16


NPSHYD     CAS-BEU01    -> earliest data ~ 2019-01-31


,ngc_bmu_id,fuel_type,earliest_data_date
0,ABRBO-1,WIND,2019-01-31
1,BAGE-1,CCGT,NaN
2,DNGB21,NUCLEAR,2019-01-31
3,DRAXX-5,COAL,2019-01-31
4,DNBAR-1,BIOMASS,2021-02-16
5,CAS-BEU01,NPSHYD,2019-01-31


In [11]:
history_df.to_csv(REF_DIR / "sample_bmu_history_depth.csv", index=False)

### Sanity-checking two surprises in the result above

Two things in `history_df` don't match the naive "wind newer, thermal older" expectation, so worth checking they're real rather than a bug in the probe:

1. Wind, nuclear, coal and hydro all bottomed out at the **same date** (2019-01-31), regardless of fuel type.
2. The CCGT sample (`BAGE-1`) returned **no data at all**, even for a recent date.

In [12]:
# 1. Is DRAXX-5's cutoff a hard boundary, or did the binary search land on a fluke gap?
around_cutoff = requests.get(
    f"{ELEXON_API}/datasets/B1610/stream",
    params={"from": "2019-01-25", "to": "2019-02-05", "bmUnit": "DRAXX-5"},
    timeout=30,
).json()
dates_around_cutoff = sorted(set(r["settlementDate"] for r in around_cutoff))
print(f"DRAXX-5 records 2019-01-25 to 2019-02-05: {len(around_cutoff)}, covering dates {dates_around_cutoff}")

# 2. Is BAGE-1 genuinely empty right now, and is it still a registered live BMU?
recent_bage = requests.get(
    f"{ELEXON_API}/datasets/B1610/stream",
    params={"from": "2026-07-01", "to": "2026-07-05", "bmUnit": "BAGE-1"},
    timeout=30,
).json()
bage_ref = bmunits[bmunits["nationalGridBmUnit"] == "BAGE-1"]
print(f"BAGE-1 recent (2026-07-01 to 2026-07-05) record count: {len(recent_bage)}")
print("BAGE-1 in reference/bmunits/all:")
bage_ref[["nationalGridBmUnit", "fuelType", "leadPartyName", "generationCapacity"]]

DRAXX-5 records 2019-01-25 to 2019-02-05: 193, covering dates ['2019-02-01', '2019-02-02', '2019-02-03', '2019-02-04', '2019-02-05']


BAGE-1 recent (2026-07-01 to 2026-07-05) record count: 0
BAGE-1 in reference/bmunits/all:


,nationalGridBmUnit,fuelType,leadPartyName,generationCapacity
417,BAGE-1,CCGT,Baglan Operations Ltd,500.000


## Step 5 — Findings summary

**Column names actually returned:**

| Source | Key columns |
|---|---|
| OSUKED `fuel_types` | `ngc_bmu_id`, `fuel_type`, `comments` |
| OSUKED `plant_locations` | `dictionary_id`, `longitude`, `latitude` |
| OSUKED `dictionary_ids` | `dictionary_id`, `name`, `ngc_bmu_id` (comma-list), `sett_bmu_id` (comma-list), plus various external-id columns not needed here |
| Elexon `reference/bmunits/all` | `nationalGridBmUnit`, `elexonBmUnit`, `fuelType`, `bmUnitName`, `leadPartyName`, `bmUnitType`, `productionOrConsumptionFlag`, `generationCapacity`, `demandCapacity`, ... (22 columns total) |
| Elexon `datasets/B1610/stream` | `dataset`, `psrType`, `bmUnit` (Elexon BMU id), `nationalGridBmUnitId`, `settlementDate`, `settlementPeriod`, `halfHourEndTime`, `quantity` (MWh) |

**Join success rate (live numbers from this run):**

- OSUKED `plant_locations`: 243 sites. `dictionary_ids`: 277 site rows, exploding to 444 individual BMU rows once the comma-separated `ngc_bmu_id` lists are split.
- `plant_locations -> dictionary_ids -> fuel_types` join: **403 BMU rows successfully linked**, spanning **213 distinct sites** (of 243 — most sites link, some don't have a matching fuel-type entry or exploded BMU that survives the join).
- Elexon's live `reference/bmunits/all`: **3,052 distinct BMUs**. OSUKED `fuel_types`: **462 distinct BMUs**.
- Overlap: **397 BMUs in both**. **65 BMUs only in OSUKED** (not in Elexon's current live list — likely decommissioned/renamed, e.g. old Aberthaw/Barking units). **2,655 BMUs only in Elexon** (no OSUKED fuel-type mapping — mostly small/embedded generation never catalogued by OSUKED).
- Elexon's own `fuelType` field (on `bmunits/all`) is populated for only **579 of 3,053 rows** — so it can plug some of the 2,655 OSUKED-gap, but not most of it.

**History depth by fuel type (6-BMU cross-fuel-type sample, day-level binary search):**

| Fuel type | BMU | Earliest data found |
|---|---|---|
| WIND | ABRBO-1 | 2019-01-31 |
| NUCLEAR | DNGB21 | 2019-01-31 |
| COAL | DRAXX-5 | 2019-01-31 |
| NPSHYD | CAS-BEU01 | 2019-01-31 |
| BIOMASS | DNBAR-1 | 2021-02-16 |
| CCGT | BAGE-1 | no data at all, even recently |

This does **not** match the naive expectation of "wind from ~2014-15, thermal further back" — instead, four completely different fuel types all bottom out on the *same* date. Sanity-checked directly (see cell above): DRAXX-5 (coal) has zero B1610 records for 2019-01-25 to 2019-01-31 and data resumes cleanly from 2019-02-01, so this is a real boundary, not a probe bug. Reading it alongside Elexon's own documentation quote ("Elexon is back-filling historical data... by January 2024 they expect to access older data sets further back than June 2023") — this looks like a **platform-wide Insights Solution backfill floor around early 2019**, not a per-site commissioning date. Older history for these BMUs may exist only via the legacy BMRS API, which is out of scope for this exploration.

BAGE-1 (Baglan CCGT) is registered in `reference/bmunits/all` with a 500 MW generation capacity but returns **zero B1610 records**, including for a recent week — sanity-checked directly above. This reads as a mothballed/inactive plant that's still on the reference list but not actually delivering metered data, not a bug in the query.

**Caveats for Simon to weigh before deciding filters:**
- The `ngc_bmu_id`/`sett_bmu_id` one-to-many relationship in `dictionary_ids` means "one site" and "one BMU" are not the same unit of analysis — worth deciding up front which the pipeline will key on.
- The apparent ~2019-02-01 backfill floor (if it holds across the full fleet, not just this sample) caps how far back a B1610-only pipeline can go, regardless of fuel type or plant age.
- Registered BMUs can have zero B1610 data (mothballed capacity, like BAGE-1) — the extraction design needs to tolerate and log empty results rather than treating them as errors.
- `only_osuked` BMUs (in OSUKED but not in Elexon's live list) are likely decommissioned or renamed and probably shouldn't be extracted.
- `only_elexon` BMUs (live but with no OSUKED fuel-type mapping) can partly use Elexon's own `fuelType` field, but it only covers 579/3,053 rows — most would need a different fuel-type source or exclusion.

## Step 6 — Proposed resumable extraction design (for review, not built)

**This is a proposal only.** Nothing below is implemented in this repo yet — it's flagged here for Simon to review and adjust before any full-fleet pull starts.

- **One raw file per BMU**, under `data/raw/generation/{ngc_bmu_id}.csv` (or `.parquet` — worth deciding once row counts from a full pull are known; CSV is easier to inspect mid-run, parquet is smaller and faster to reload).
- **A manifest file**, e.g. `data/interim/extraction_manifest.csv`, with one row per `(bmu, date_range_chunk)` attempted, columns roughly: `ngc_bmu_id`, `chunk_start`, `chunk_end`, `status` (`pending`/`success`/`failed`), `rows_returned`, `attempted_at`, `error`. Chunking by, say, one calendar year per row keeps individual requests small and retries cheap.
- **Resume logic**: before each run, load the manifest, skip any `(bmu, chunk)` already marked `success`, retry `failed`, and only issue fresh requests for chunks not yet attempted. This means an interrupted run (network drop, rate limit, laptop sleep) can restart from exactly where it left off rather than re-pulling everything.
- **Rate/politeness**: no API key or documented rate limit was hit during this exploration, but a full-fleet multi-year pull is a much larger volume of requests — worth adding a small delay between calls and treating HTTP 429/5xx as retryable rather than fatal in the manifest.
- **Open questions for Simon to decide, not assumed here**: which BMUs to include (fuel-type filter? exclude interconnectors/demand-only units?), a minimum history-length cutoff, whether to key the final dataset on BMU or on site (`dictionary_id`), and CSV vs parquet for the raw store.